- pip install Pillow
- pip install torch
- pip install tqdm
- pip install wildlife_tools
- pip install

In [1]:
import sys
import torch

print("Python :", sys.executable)
print("Torch :", torch.__version__)
print("Torch CUDA :", torch.version.cuda)
print("CUDA available :", torch.cuda.is_available())
print("GPU count :", torch.cuda.device_count())

Python : c:\HyeonKyu\for_mate\.venv\Scripts\python.exe
Torch : 2.13.0+cu130
Torch CUDA : 13.0
CUDA available : True
GPU count : 1


In [2]:
from pathlib import Path
import numpy as np
import torch
from PIL import Image
from tqdm import tqdm

from wildlife_tools.similarity import CosineSimilarity
from wildlife_tools.features import DeepFeatures
from wildlife_tools.data import WildlifeDataset
from wildlife_tools.train import ArcFaceLoss

import timm
from torchvision import transforms


# =========================
# 1. 경로 설정
# =========================
DATA_DIR = Path(r"./processed_animals")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("DEVICE:", DEVICE)


# =========================
# 2. MegaDescriptor 불러오기
# =========================
model = timm.create_model(
    "hf-hub:BVRA/MegaDescriptor-L-384",
    pretrained=True
)

model = model.to(DEVICE)
model.eval()


# =========================
# 3. MegaDescriptor 전처리
# =========================
transform = transforms.Compose([
    transforms.Resize((384, 384)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


# =========================
# 4. 이미지 경로 수집
# =========================
image_paths = []
labels = []

extensions = {".jpg", ".jpeg", ".png", ".webp"}

for animal_dir in sorted(DATA_DIR.iterdir()):

    if not animal_dir.is_dir():
        continue

    animal_id = animal_dir.name

    for image_path in animal_dir.iterdir():

        if image_path.suffix.lower() in extensions:
            image_paths.append(image_path)
            labels.append(animal_id)


print("개체 수:", len(set(labels)))
print("이미지 수:", len(image_paths))


# =========================
# 5. 이미지 → embedding
# =========================
embeddings = []

with torch.no_grad():

    for image_path in tqdm(image_paths, desc="Embedding 추출"):

        image = Image.open(image_path).convert("RGB")

        image = transform(image)
        image = image.unsqueeze(0).to(DEVICE)

        embedding = model(image)

        # L2 Normalization
        embedding = torch.nn.functional.normalize(
            embedding,
            p=2,
            dim=1
        )

        embeddings.append(
            embedding.cpu().numpy()[0]
        )


embeddings = np.array(embeddings)

print("Embedding shape:", embeddings.shape)


# =========================
# 6. embedding 저장
# =========================
np.save("megadescriptor_embeddings.npy", embeddings)
np.save("megadescriptor_labels.npy", np.array(labels))
np.save(
    "megadescriptor_paths.npy",
    np.array([str(p) for p in image_paths])
)

print("embedding 저장 완료")


# =========================
# 7. Cosine Similarity Matrix
# =========================
# L2 normalize 되어 있으므로
# dot product = cosine similarity

similarity_matrix = embeddings @ embeddings.T

print("Similarity matrix:", similarity_matrix.shape)


# =========================
# 8. Top-1 / Top-5 평가
# =========================
top1_correct = 0
top5_correct = 0

num_queries = len(image_paths)

for i in range(num_queries):

    similarities = similarity_matrix[i].copy()

    # 자기 자신은 검색 결과에서 제외
    similarities[i] = -np.inf

    # similarity 높은 순서
    ranking = np.argsort(similarities)[::-1]

    query_label = labels[i]

    # Top-1
    top1_index = ranking[0]

    if labels[top1_index] == query_label:
        top1_correct += 1

    # Top-5
    top5_indices = ranking[:5]

    if any(labels[idx] == query_label for idx in top5_indices):
        top5_correct += 1


top1_accuracy = top1_correct / num_queries
top5_accuracy = top5_correct / num_queries


print()
print("======================")
print("MegaDescriptor 결과")
print("======================")

print(f"Query 이미지 수 : {num_queries}")
print(f"Top-1 Accuracy : {top1_accuracy:.4f}")
print(f"Top-5 Accuracy : {top5_accuracy:.4f}")


# =========================
# 9. 동일 / 다른 개체 similarity
# =========================
same_scores = []
different_scores = []

for i in range(len(image_paths)):

    for j in range(i + 1, len(image_paths)):

        score = similarity_matrix[i, j]

        if labels[i] == labels[j]:
            same_scores.append(score)

        else:
            different_scores.append(score)


same_mean = np.mean(same_scores)
different_mean = np.mean(different_scores)

print()
print("======================")
print("Similarity 비교")
print("======================")

print(
    f"동일 개체 평균 similarity : "
    f"{same_mean:.4f}"
)

print(
    f"다른 개체 평균 similarity : "
    f"{different_mean:.4f}"
)

c:\HyeonKyu\for_mate\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DEVICE: cuda


개체 수: 344
이미지 수: 1358


Embedding 추출: 100%|██████████| 1358/1358 [00:45<00:00, 30.10it/s]


Embedding shape: (1358, 1536)
embedding 저장 완료
Similarity matrix: (1358, 1358)

MegaDescriptor 결과
Query 이미지 수 : 1358
Top-1 Accuracy : 0.7172
Top-5 Accuracy : 0.8586

Similarity 비교
동일 개체 평균 similarity : 0.4096
다른 개체 평균 similarity : 0.0870


배경 제거한 데이터 셋과 비교

In [3]:
from torch.utils.data import Dataset, DataLoader


class FlatImageDataset(Dataset):
    def __init__(self, root, transform):
        self.transform = transform
        self.samples = []
        exts = {".jpg", ".jpeg", ".png", ".webp"}
        for d in sorted(p for p in Path(root).iterdir() if p.is_dir()):
            for f in sorted(d.iterdir()):
                if f.suffix.lower() in exts:
                    self.samples.append((f, d.name))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, i):
        path, label = self.samples[i]
        img = Image.open(path).convert("RGB")
        return self.transform(img), label


@torch.no_grad()
def extract_embeddings(root, batch_size=32):
    ds = FlatImageDataset(root, transform)
    dl = DataLoader(ds, batch_size=batch_size, num_workers=0)

    embs, labels = [], []
    for x, lab in tqdm(dl, desc=f"embedding: {root}"):
        out = model(x.to(DEVICE))
        out = torch.nn.functional.normalize(out, p=2, dim=1)
        embs.append(out.cpu().numpy())
        labels.extend(lab)

    return np.concatenate(embs), np.array(labels)


def evaluate_loo(embeddings, labels):
    """전체 leave-one-out (기존 방식)."""
    labels = np.asarray(labels)
    sim = embeddings @ embeddings.T
    np.fill_diagonal(sim, -np.inf)

    n = len(labels)
    top1 = top5 = 0
    for i in range(n):
        ranking = np.argsort(sim[i])[::-1]
        if labels[ranking[0]] == labels[i]:
            top1 += 1
        if np.any(labels[ranking[:5]] == labels[i]):
            top5 += 1
    return top1 / n, top5 / n


def evaluate_split(embeddings, labels, seed=0):
    """개체별 1장만 gallery, 나머지는 query — 더 현실적인 검색 세팅."""
    labels = np.asarray(labels)
    rng = np.random.default_rng(seed)

    gallery_idx = np.array([
        rng.choice(np.where(labels == lab)[0])
        for lab in np.unique(labels)
    ])
    g_set = set(gallery_idx.tolist())
    query_idx = np.array([i for i in range(len(labels)) if i not in g_set])

    g_emb, g_lab = embeddings[gallery_idx], labels[gallery_idx]
    sim = embeddings[query_idx] @ g_emb.T

    top1 = top5 = 0
    for row, gt in zip(sim, labels[query_idx]):
        ranking = np.argsort(row)[::-1]
        if g_lab[ranking[0]] == gt:
            top1 += 1
        if np.any(g_lab[ranking[:5]] == gt):
            top5 += 1
    q = len(query_idx)
    return top1 / q, top5 / q

In [4]:
emb_orig, lab_orig = extract_embeddings("processed_animals")
emb_mask, lab_mask = extract_embeddings("processed_animals_masked")

for name, e, l in [("원본", emb_orig, lab_orig), ("배경제거", emb_mask, lab_mask)]:
    t1, t5 = evaluate_loo(e, l)
    s1, s5 = evaluate_split(e, l)
    print(f"[{name}]")
    print(f"  LOO    Top-1 {t1:.4f} / Top-5 {t5:.4f}")
    print(f"  split  Top-1 {s1:.4f} / Top-5 {s5:.4f}")
    print(f"  개체 {len(set(l))} / 이미지 {len(l)}\n")

embedding: processed_animals_masked: 100%|██████████| 42/42 [00:42<00:00,  1.01s/it]

[원본]
  LOO    Top-1 0.7165 / Top-5 0.8586
  split  Top-1 0.5276 / Top-5 0.7337
  개체 344 / 이미지 1358

[배경제거]
  LOO    Top-1 0.6443 / Top-5 0.8142
  split  Top-1 0.4338 / Top-5 0.6507
  개체 331 / 이미지 1313

